# GCP Infra, DevOps & Reliability — Hands-On Lab
### GCP Training Program — Day 5, Module 8: GCP Infra, DevOps & Reliability

**What this notebook covers:** live Monitoring & Logging, LLM connectivity patterns from GCP
(including Claude via Vertex AI Model Garden's partner models), and guided walkthroughs of ML on
GKE and multi-project topology.

**Note on scope:** the agenda line "GCP connectivity with Claude / LLMs" is a bit open to
interpretation — I've read it as *how to securely call an external/partner LLM API from GCP
infrastructure* (credential handling via Secret Manager, and Claude's specific availability as a
Vertex AI Model Garden partner model). Adjust Section 5 if you had something more specific in mind.

## ⚠️ Cost & trial-account notes
**GKE is the one genuinely expensive resource on this agenda.** Even the smallest Autopilot cluster
takes 5-10 minutes to provision and bills continuously once created. Section 6 is a **guided
walkthrough with optional live execution** — read the cost note there before deciding whether to
run it in front of a trial-account class. Multi-project topology (Section 7) needs multiple
projects and org-level permissions most trial accounts don't have, so it's diagram/walkthrough
only, not executable here regardless.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs
against your own project.


## 1. Authenticate This Session

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Configure Your Project & Region

In [ ]:
import time
import os

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}
!gcloud services enable logging.googleapis.com monitoring.googleapis.com secretmanager.googleapis.com --project={PROJECT_ID}
print("Project configured:", PROJECT_ID)

## 3. Monitoring & Logging
### 3.1 Write & Read a Custom Log Entry
**What this does:** writes a structured log entry programmatically (the same mechanism any
application would use), then reads it back with a filter — the basic read/write loop behind every
more advanced logging feature.

In [ ]:
!pip install --quiet google-cloud-logging google-cloud-monitoring

In [ ]:
from google.cloud import logging as gcp_logging

logging_client = gcp_logging.Client(project=PROJECT_ID)
logger = logging_client.logger("training-demo-log")

logger.log_struct({
    "message": "Training demo log entry",
    "severity_context": "demo",
    "component": "gcp-infra-lab",
})
print("Wrote a structured log entry to 'training-demo-log'.")

In [ ]:
!gcloud logging read "logName=projects/{PROJECT_ID}/logs/training-demo-log" --project={PROJECT_ID} --limit=5 --format=json

### 3.2 Create a Log-Based Metric
**What this does:** turns a log filter into a numeric time-series metric — every future log entry
matching the filter increments a counter you can chart, alert on, or dashboard, instead of manually
grepping logs after the fact.

In [ ]:
METRIC_NAME = f"training-demo-metric-{_suffix}"

!gcloud logging metrics create {METRIC_NAME} \
    --project={PROJECT_ID} \
    --description="Counts training-demo-log entries" \
    --log-filter='logName="projects/{PROJECT_ID}/logs/training-demo-log"'

### 3.3 Create an Alert Policy on That Metric
**What this does:** wires the metric above to an alert condition — here, "fire if the count
exceeds 0 in a 60-second window," a deliberately trivial threshold so it's easy to trigger live for
demo purposes (a real policy would use a meaningful threshold and duration).

In [ ]:
import json

alert_policy = {
    "displayName": f"training-demo-alert-{_suffix}",
    "conditions": [
        {
            "displayName": "Log entries present",
            "conditionThreshold": {
                "filter": f'metric.type="logging.googleapis.com/user/{METRIC_NAME}" AND resource.type="global"',
                "comparison": "COMPARISON_GT",
                "thresholdValue": 0,
                "duration": "60s",
                "aggregations": [{"alignmentPeriod": "60s", "perSeriesAligner": "ALIGN_COUNT"}],
            },
        }
    ],
    "combiner": "OR",
    "notificationChannels": [],
}

with open("alert_policy.json", "w") as f:
    json.dump(alert_policy, f)

print("Wrote alert_policy.json — note: no notificationChannels configured, so this creates the")
print("policy without actually paging anyone. Add a real channel ID for a production policy.")

In [ ]:
!gcloud alpha monitoring policies create --project={PROJECT_ID} --policy-from-file=alert_policy.json

## 4. Secret Manager — Foundation for Section 5
### 4.1 Store an API Key Securely
**What this does:** demonstrates the pattern used to hold any external credential (an LLM provider
API key, a database password) without ever putting it in code or a notebook cell in plain text —
code references the *secret name*, and IAM controls who/what can read the actual value.

In [ ]:
SECRET_ID = f"demo-llm-api-key-{_suffix}"

api_key_value = input("Paste any placeholder value to store as a demo secret (not a real key needed): ").strip()

!echo -n "{api_key_value}" | gcloud secrets create {SECRET_ID} --project={PROJECT_ID} --data-file=- --replication-policy=automatic

### 4.2 Read the Secret Back Programmatically

In [ ]:
!pip install --quiet google-cloud-secret-manager

In [ ]:
from google.cloud import secretmanager

sm_client = secretmanager.SecretManagerServiceClient()
secret_path = f"projects/{PROJECT_ID}/secrets/{SECRET_ID}/versions/latest"

response = sm_client.access_secret_version(request={"name": secret_path})
retrieved_value = response.payload.data.decode("UTF-8")
print("Retrieved secret value:", retrieved_value)
print("\nIn a real application, this is exactly how you'd fetch an LLM provider API key at runtime")
print("instead of hardcoding it — combine with IAM (roles/secretmanager.secretAccessor) scoped to")
print("only the service account that needs it.")

## 5. GCP Connectivity With Claude & Other LLMs
### 5.1 Claude on Vertex AI Model Garden (Partner Model)
**What this is:** Anthropic's Claude models are available as a **partner model** in Vertex AI Model
Garden in supported regions — this means GCP-native billing, IAM, and networking apply, without
going through Anthropic's API directly.

**Caveat for the class:** partner model availability, supported regions, and the exact SDK/endpoint
pattern change over time — confirm current availability and syntax against GCP's live documentation
before presenting this as definitive. The shape below is illustrative of the pattern, not guaranteed
to match the exact current API without a docs check.

In [ ]:
# Illustrative pattern for calling a Claude model via Vertex AI Model Garden partner models.
# Confirm exact model IDs/region availability against current Vertex AI documentation before
# using this live — partner model surfaces are one of the faster-evolving areas of the platform.

# !pip install --quiet "anthropic[vertex]"
#
# from anthropic import AnthropicVertex
#
# client = AnthropicVertex(project_id=PROJECT_ID, region=REGION)
# message = client.messages.create(
#     model="claude-sonnet-4@20250514",  # confirm the current partner-model ID string in docs
#     max_tokens=256,
#     messages=[{"role": "user", "content": "Explain GCP IAM in two sentences."}],
# )
# print(message.content)

print("Reference pattern above — verify current model IDs/region support before running live.")

### 5.2 The General Connectivity Pattern (Applies to Any External LLM Provider)
Whether calling Claude via Model Garden, Anthropic's own API directly, or any other external model
provider from GCP infrastructure, the same three concerns apply — worth stating explicitly to the
class as the reusable pattern, independent of which specific provider they end up using:

1. **Credential storage** — Secret Manager (Section 4), never hardcoded, never committed to a repo.
2. **Network egress** — if calling an external API (not a GCP-native partner model endpoint) from a
   VPC with restricted egress, a Cloud NAT or explicit firewall egress rule is needed; fully private
   GKE/Compute workloads with no NAT cannot reach the public internet at all by default.
3. **IAM scoping** — grant the calling service account only `roles/secretmanager.secretAccessor` on
   the specific secret, not project-wide Secret Manager access — least privilege, same principle as
   Day 3's security module.

In [ ]:
sm_client.delete_secret(request={"name": f"projects/{PROJECT_ID}/secrets/{SECRET_ID}"})
print("Deleted demo secret — cleaning up now since Section 5 only needed it as an illustration.")

## 6. ML on GKE (Guided Walkthrough — Optional Live Execution)
**Cost note, read before deciding whether to run this live:** even the smallest GKE Autopilot
cluster takes 5-10 minutes to provision and bills continuously from the moment it's created —
there is no smaller "trial-safe" tier the way there is for a single BigQuery query or Cloud
Storage bucket. **Recommendation: walk through the commands and concept without running them live
in a trial-account class**, unless you've budgeted specifically for it and are ready to delete the
cluster within a few minutes of creating it.

**The concept:** GKE Autopilot removes node-management entirely (Google manages node provisioning,
you only think in terms of pods/deployments) — the natural choice for a training demo *if* you do
run it live, since there's no node pool sizing decision to explain or get wrong.

In [ ]:
CLUSTER_NAME = f"training-demo-cluster-{_suffix}"

# OPTIONAL — only uncomment and run if you've decided to demo a real cluster live.
# Autopilot auto-sizes nodes; there is no smaller manual machine-type choice to make here.

# !gcloud container clusters create-auto {CLUSTER_NAME} \
#     --project={PROJECT_ID} \
#     --region={REGION}

print("Cluster creation intentionally commented out — see the cost note above before uncommenting.")

In [ ]:
# A minimal deployment you'd apply once the cluster above is live — a single small container,
# standing in for a lightweight model-serving container in a real ML-on-GKE setup.

deployment_yaml = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: demo-model-server
spec:
  replicas: 1
  selector:
    matchLabels:
      app: demo-model-server
  template:
    metadata:
      labels:
        app: demo-model-server
    spec:
      containers:
        - name: demo-model-server
          image: gcr.io/google-samples/hello-app:1.0
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
"""

with open("deployment.yaml", "w") as f:
    f.write(deployment_yaml)

print(deployment_yaml)
print("Apply with: kubectl apply -f deployment.yaml   (after connecting kubectl to the cluster via")
print("gcloud container clusters get-credentials)")

In [ ]:
# Matching cleanup — run immediately after the demo if you created the cluster above.
# !gcloud container clusters delete {CLUSTER_NAME} --project={PROJECT_ID} --region={REGION} --quiet

print("Cluster deletion intentionally commented out — matches the optional creation cell above.")

## 7. Multi-Project Topology (Guided Walkthrough Only)
**Why this is walkthrough-only, not executable in this notebook:** Shared VPC and VPC Service
Controls both require multiple projects and org-level (or folder-level) IAM permissions that a
single trial-account project doesn't have — there's nothing meaningful to run here regardless of
budget. Present this as architecture/diagram content:

- **Shared VPC** — one "host" project owns the VPC network; other "service" projects attach to it
  and deploy resources using its subnets, while keeping their own billing, IAM, and resource
  ownership separate. Common pattern: a central networking team owns the host project; individual
  product teams own service projects.
- **VPC Service Controls** — draws a security perimeter *around* a set of projects/services (e.g.
  "BigQuery and GCS in these 3 projects"), blocking data exfiltration even by someone with valid
  IAM credentials, if they're accessing from outside the defined perimeter (e.g. a personal Gmail
  session vs. a corporate-network session).
- **When to bring this up relative to Day 3's security module:** IAM controls *who* can do *what*;
  Shared VPC controls *network topology*; VPC Service Controls adds a perimeter *on top of* both —
  worth being explicit that these are layered, not alternatives to each other.